# Time Series Forecasting — Colab Demo

Benchmark for **TimesNet, TimeMixer, PatchTST and ModernTCN** on ETT datasets.

| Runtime | ETTh1 · pred=96 · 10 epochs |
|---------|------------------------------|
| T4 GPU  | ~10–20 min                   |
| CPU     | ~4–8 h                       |

> **Tip:** Runtime → Change runtime type → T4 GPU

In [1]:
!nvidia-smi

Wed May 20 12:19:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

True
Tesla T4


In [3]:
!git clone https://github.com/caltinuzengi/Time-Series-Library.git
%cd Time-Series-Library/

Cloning into 'Time-Series-Library'...
remote: Enumerating objects: 146, done.
remote: Counting objects: 100% (146/146), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 146 (delta 59), reused 114 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (146/146), 183.44 KiB | 469.00 KiB/s, done.
Resolving deltas: 100% (59/59), done.
/content/Time-Series-Library


In [ ]:
!git pull
!pip install uv -q
!uv sync

In [ ]:
!uv run download-data

In [ ]:
!bash scripts/TimesNet/debug_ETTh1.sh

In [ ]:
!bash scripts/TimeMixer/debug_ETTh1.sh

In [ ]:
!bash scripts/PatchTST/debug_ETTh1.sh

In [ ]:
!bash scripts/ModernTCN/debug_ETTh1.sh

In [ ]:
!bash scripts/TimesNet/benchmark_forecasting.sh

In [ ]:
!bash scripts/TimeMixer/benchmark_forecasting.sh

In [ ]:
!bash scripts/PatchTST/benchmark_forecasting.sh

In [ ]:
!bash scripts/ModernTCN/benchmark_forecasting.sh

## 📊 Sonuçlar

Aşağıdaki hücreler `logs/` klasöründeki JSON dosyalarını okur — önce yukarıdaki eğitim hücrelerini çalıştırın.

In [ ]:
import json, glob
import matplotlib.pyplot as plt

models    = ["TimesNet", "TimeMixer", "PatchTST", "ModernTCN"]
pred_lens = [24, 96]
dataset   = "ETTh1"   # temsili veri seti

fig, axes = plt.subplots(len(pred_lens), len(models), figsize=(18, 7), sharey=False)

for row, pred_len in enumerate(pred_lens):
    for col, model in enumerate(models):
        ax = axes[row][col]
        files = sorted(glob.glob(f"logs/{model}_{dataset}_forecasting_{pred_len}_*.json"))
        if not files:
            ax.text(0.5, 0.5, "log yok", ha="center", va="center",
                    transform=ax.transAxes, color="gray", fontsize=10)
            if row == 0:
                ax.set_title(model, fontsize=11)
            continue
        with open(files[-1]) as f:
            d = json.load(f)
        logs = d["epoch_logs"]
        epochs = [e["epoch"] for e in logs]
        ax.plot(epochs, [e["train_loss"] for e in logs], "o-",  label="train", ms=4, lw=1.5)
        ax.plot(epochs, [e["val_loss"]   for e in logs], "s--", label="val",   ms=4, lw=1.5)
        if row == 0:
            ax.set_title(model, fontsize=11)
        ax.set_xlabel("Epoch", fontsize=8)
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)
        if col == 0:
            ax.set_ylabel(f"pred={pred_len}\nMSE", fontsize=9)

plt.suptitle(f"{dataset} — Training Curves (pred_len 24 vs 96)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
import json, glob, pandas as pd

rows = []
for path in sorted(glob.glob("results/*_forecasting_*.json")):
    with open(path) as f:
        d = json.load(f)
    cfg, met = d["config"], d["metrics"]
    rows.append({
        "Model":    cfg.get("model",    "-"),
        "Data":     cfg.get("data",     "-"),
        "pred_len": cfg.get("pred_len", "-"),
        "MSE":  round(met.get("mse",  float("nan")), 4),
        "MAE":  round(met.get("mae",  float("nan")), 4),
        "RMSE": round(met.get("rmse", float("nan")), 4),
    })

if rows:
    df = pd.DataFrame(rows).sort_values(["Data", "pred_len", "MSE"])
    display(df)
else:
    print("Henüz sonuç yok — önce eğitim hücrelerini çalıştırın.")